# GoogLeNet / Inception — Going Deeper with Convolutions (2014)

_Papers · Computer Vision_

**Run 1×1, 3×3, 5×5, and pooling in parallel and concatenate them — with 1×1 bottlenecks to keep it cheap.**

---

This notebook is a practice scaffold for the **GoogLeNet / Inception — Going Deeper with Convolutions (2014)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# --- 0. Worked example: the 1x1 bottleneck's FLOP/param saving on the 5x5 branch. ---
# Module (3a): H=W=28, C_in=192, 5x5 branch outputs C_out=32, reduce r=16  (Table 1).
H, W, Cin, Cout, r = 28, 28, 192, 32, 16
naive_w = 5*5 * Cin * Cout                      # one 5x5: 192 -> 32
bott_w  = 1*1 * Cin * r + 5*5 * r * Cout        # 1x1 (192->16) then 5x5 (16->32)
naive_macs = H*W * naive_w
bott_macs  = H*W * bott_w
print("5x5 branch  NAIVE   weights =", naive_w, " MACs =", naive_macs)
print("5x5 branch  BOTTLE  weights =", bott_w,  " MACs =", bott_macs)
print("weight saving =", round(naive_w/bott_w, 2), "x   MAC saving =", round(naive_macs/bott_macs, 2), "x")
# 5x5 branch  NAIVE   weights = 153600  MACs = 120422400
# 5x5 branch  BOTTLE  weights = 15872   MACs = 12443648
# weight saving = 9.68 x   MAC saving = 9.68 x


# --- 1. The dimension-reduced Inception module (built by hand). ---
class Inception(nn.Module):
    def __init__(self, Cin, c1, c3r, c3, c5r, c5, pp):
        super().__init__()
        # branch 1: a single 1x1 conv
        self.b1 = nn.Conv2d(Cin, c1, kernel_size=1)
        # branch 2: 1x1 REDUCE -> 3x3 (same-padding=1)
        self.b3 = nn.Sequential(
            nn.Conv2d(Cin, c3r, 1), nn.ReLU(inplace=True),
            nn.Conv2d(c3r, c3, 3, padding=1))
        # branch 3: 1x1 REDUCE -> 5x5 (same-padding=2)
        self.b5 = nn.Sequential(
            nn.Conv2d(Cin, c5r, 1), nn.ReLU(inplace=True),
            nn.Conv2d(c5r, c5, 5, padding=2))
        # branch 4: 3x3 max-pool (stride 1, padding 1) -> 1x1 project
        self.bp = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(Cin, pp, 1))

    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)  # channel axis


# --- 2. Module (3a) from Table 1: Cin=192, 1x1=64, (3x3r=96)3x3=128, (5x5r=16)5x5=32, pool-proj=32. ---
m3a = Inception(192, c1=64, c3r=96, c3=128, c5r=16, c5=32, pp=32)
x = torch.randn(2, 192, 28, 28)                 # (N, C, H, W)
y = m3a(x)
print("\ninput :", tuple(x.shape))
print("output:", tuple(y.shape), "  (channels = 64+128+32+32 =", 64+128+32+32, ")")
assert y.shape == (2, 256, 28, 28), "concat shape must be 28x28x256"
print("shape check passed: parallel branches concatenated on the channel axis.")
params_reduced = sum(p.numel() for p in m3a.parameters())
print("dim-reduced module (3a) params (incl. bias):", params_reduced)


# --- 3. ABLATION: the SAME module without 1x1 reduces (3x3/5x5 read all 192 channels). ---
class NaiveInception(nn.Module):
    def __init__(self, Cin, c1, c3, c5, pp):
        super().__init__()
        self.b1 = nn.Conv2d(Cin, c1, 1)
        self.b3 = nn.Conv2d(Cin, c3, 3, padding=1)        # no reduce -> 192 -> 128 directly
        self.b5 = nn.Conv2d(Cin, c5, 5, padding=2)        # no reduce -> 192 -> 32 directly
        self.bp = nn.Sequential(nn.MaxPool2d(3, 1, 1), nn.Conv2d(Cin, pp, 1))
    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)

m3a_naive = NaiveInception(192, c1=64, c3=128, c5=32, pp=32)
y_naive = m3a_naive(x)
params_naive = sum(p.numel() for p in m3a_naive.parameters())
print("\nABLATION (no 1x1 reduces):")
print("output:", tuple(y_naive.shape), " -> SAME 256 channels, so the interface is unchanged")
print("naive module (3a) params (incl. bias):", params_naive)
print("whole-module param blow-up without bottlenecks =",
      round(params_naive / params_reduced, 2), "x")
# The output shape is identical; only the parameter/MAC cost changed. The 1x1 reduces are a
# pure efficiency win -- the paper's reason GoogLeNet could be 22 layers deep cheaply.
# (Our exact arithmetic from the channel counts, not a headline number from the paper.)

## Visualize the data & results

_Per branch, how many parameters does the 1×1 bottleneck save versus a naive Inception module on module (3a)'s channel counts?_

In [ ]:
# Exact parameter counts from the convolution cost formula: weights = k^2 * C_in * C_out.
# Module (3a), C_in = 192 (Table 1).
def conv_w(k, cin, cout): return k*k*cin*cout

Cin = 192
# 5x5 branch: naive 192->32  vs  bottleneck 1x1(192->16) + 5x5(16->32)
n5 = conv_w(5, Cin, 32)
b5 = conv_w(1, Cin, 16) + conv_w(5, 16, 32)
# 3x3 branch: naive 192->128 vs  bottleneck 1x1(192->96) + 3x3(96->128)
n3 = conv_w(3, Cin, 128)
b3 = conv_w(1, Cin, 96) + conv_w(3, 96, 128)

print("5x5  naive =", n5, " bottleneck =", b5, " ratio =", round(n5/b5, 2))
print("3x3  naive =", n3, " bottleneck =", b3, " ratio =", round(n3/b3, 2))
# 5x5  naive = 153600  bottleneck = 15872  ratio = 9.68
# 3x3  naive = 221184  bottleneck = 129024 ratio = 1.71
# (thousands, for the bars: 153.6 / 15.872 and 221.184 / 129.024)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Take working module (3a) and remove the $1\times1$ reduces (so the
            $3\times3$ and $5\times5$ read all 192 input channels directly, and the pool branch keeps all 192).
            The concatenated output is still 256 channels. So what actually changed, and what does that
            demonstrate about the bottleneck's role?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Keep every branch's output channel count the same (1×1=64, 3×3=128, 5×5=32, pool-proj=32), so the concat is still 256 — the shape is unchanged. — _An honest ablation changes one thing: the presence of the 1×1 reduces, not the module's output interface._
- Count parameters/MACs for both versions and compare. The naive 5×5 alone jumps from 15,872 to 153,600 weights; the 3×3 and pool-proj also grow. — _Removing the reduce forces the expensive spatial convs to run on the full 192 input channels — exactly the 'prohibitively expensive' blow-up the paper warns about._
- Conclude the bottleneck did not change what the module outputs, only how cheaply it computes it. — _Same output shape, far fewer weights/MACs — the 1×1 reduce is a pure efficiency win, which is the paper's point._

**Answer:** The output shape is unchanged (still $28\times28\times256$), but the parameter and MAC counts
                 jump sharply &mdash; the $5\times5$ branch alone goes from $\sim\!15.9$k to $153.6$k weights
                 ($\approx 9.68\times$). This isolates the $1\times1$ reduce as a compute/parameter
                 optimization: it buys the same multi-scale output far more cheaply, which is what let GoogLeNet
                 go 22 layers deep without a parameter explosion. The CODEVIZ panel shows this per-branch.

</details>

**Problem 2.** You build the module but torch.cat throws a shape-mismatch error. The $3\times3$ branch
            output is $28\times28$ but the $5\times5$ branch output is $24\times24$. What went wrong, and how
            do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check padding: a 5×5 conv with padding=0 shrinks 28→24 (loses (k-1)=4 pixels), while the 3×3 with padding=1 stayed at 28. — _Concatenation on the channel axis requires identical H and W; a branch with too little padding comes out smaller._
- Set padding = (k-1)/2 for each branch: 0 for 1×1, 1 for 3×3, 2 for 5×5, and padding=1 on the 3×3 pool. — _Same-padding keeps every branch at 28×28 so the four outputs align spatially._
- Re-run; all four branches are now 28×28 and concatenate to 28×28×(sum of channels). — _Matching spatial size is the precondition for the channel-axis concat._

**Answer:** The $5\times5$ branch lacked padding, so it shrank to $24\times24$ while the $3\times3$ stayed
                 $28\times28$ &mdash; and torch.cat on the channel axis needs equal $H,W$. Fix it with
                 same-padding: padding $=(k-1)/2$, i.e. $2$ for the $5\times5$ (and $1$ for the $3\times3$
                 and the pool). All branches return $28\times28$ and the concat succeeds.

</details>

**Problem 3.** Repeat the worked example for the $3\times3$ branch of module (3a): naive is a $3\times3$
            from 192&rarr;128; the bottleneck is $1\times1$ (192&rarr;96) then $3\times3$ (96&rarr;128).
            Compute the weight counts and the savings ratio. Why is it smaller than the $5\times5$ branch's
            $\sim\!9.68\times$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Naive 3×3 weights: 3²·192·128 = 9·192·128 = 221,184. — _k²·C_in·C_out with k=3, C_in=192, C_out=128._
- Bottleneck: 1×1 is 1·192·96 = 18,432; 3×3 is 9·96·128 = 110,592; total = 129,024. — _The reduce trims 192→96 channels before the 3×3 runs._
- Ratio = 221,184 / 129,024 ≈ 1.71×. The reduce here only halves channels (192→96, not 192→16), and k²=9 is much smaller than 25, so the saving is milder. — _Savings scale with how aggressively you reduce C_in and with k²; the 5×5 branch reduces 12× and pays k²=25, so it gains far more._

**Answer:** Naive $3\times3$: $9\cdot192\cdot128 = 221{,}184$ weights. Bottleneck:
                 $18{,}432 + 110{,}592 = 129{,}024$. Ratio $\approx \mathbf{1.71\times}$ &mdash; real, but far less
                 than the $5\times5$ branch's $9.68\times$. Two reasons: the $3\times3$ reduce is gentle
                 ($192\to96$, only $2\times$) whereas the $5\times5$ reduce is aggressive ($192\to16$,
                 $12\times$); and $k^2=9$ for the $3\times3$ versus $25$ for the $5\times5$, so the expensive
                 layer the bottleneck shrinks is itself cheaper. The bottleneck pays off most where the kernel is
                 big and the reduce is deep.

</details>